# Topic 9b: Logistic Regression in Practice: Real Patients, by Hand and with scikit-learn

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 9b lecture note](https://dongchul.kim/csci3320-fall2026/topic/9b), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## The data: 569 real patients

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
print("X:", data.data.shape, "  (569 tumours, 30 measurements each)")
print("y:", data.target.shape)
print("\nlabel 0 =", data.target_names[0], "   label 1 =", data.target_names[1])
print("\nfirst six measurements:", [str(f) for f in data.feature_names[:6]])
print("\ncounts: malignant", (data.target == 0).sum(), " benign", (data.target == 1).sum())

### First decision: which class is `1`?

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = 1 - data.target                      # now 1 = malignant, the thing we look for

print("malignant (y = 1):", (y == 1).sum())
print("benign    (y = 0):", (y == 0).sum())
print(f"\nmalignant share: {(y == 1).mean():.1%}")

## Split before you look

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, 1 - data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

print("train:", X_train.shape, " malignant:", y_train.sum())
print("test :", X_test.shape,  " malignant:", y_test.sum())

## Example 1: the loop from Topic 9a, on real data

### First attempt, and it does not work

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

i = list(data.feature_names).index("mean radius")
x_train = X_train[:, i]                  # raw millimetres, straight from the file

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w, b = 0.0, 0.0
alpha = 0.5                              # the same learning rate as Topic 9a
for step in range(3000):
    p = sigmoid(w * x_train + b)
    w -= alpha * np.mean((p - y_train) * x_train)
    b -= alpha * np.mean(p - y_train)

p = sigmoid(w * x_train + b)
print(f"w = {w:.4f}   b = {b:.4f}")
print(f"training accuracy = {((p >= 0.5).astype(int) == y_train).mean():.4f}")

## Normalization: why, and the two ways to do it

### The same four numbers, both ways

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

x = np.array([[1.0], [2.0], [3.0], [4.0]])

print("raw           ", x.ravel())
print("min-max       ", MinMaxScaler().fit_transform(x).ravel().round(2))
print("standard score", StandardScaler().fit_transform(x).ravel().round(2))

print("\nby hand:")
v = x.ravel()
print("  min-max       ", ((v - v.min()) / (v.max() - v.min())).round(2))
print("  standard score", ((v - v.mean()) / v.std()).round(2))
print(f"\n  mean {v.mean()}, standard deviation {v.std():.2f}")

### Our thirty columns, before and after

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
Z_train = StandardScaler().fit_transform(X_train)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(X_train, showfliers=False)
ax[0].set_yscale("log")                       # a linear axis cannot show both ends
ax[0].set_title("raw: thousandths on one column, thousands on another")
ax[0].set_ylabel("value (log scale)")
ax[1].boxplot(Z_train, showfliers=False)
ax[1].axhline(0, color="#dc2626", lw=1)
ax[1].set_title("standardized: every column centred at 0, spread 1")
ax[1].set_ylabel("standard scores")
for a in ax:
    a.set_xlabel("feature number")
    a.set_xticks([1, 10, 20, 30]); a.set_xticklabels(["1", "10", "20", "30"])
plt.tight_layout()
plt.show()

### Second attempt, standardized

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

i = list(data.feature_names).index("mean radius")
x_train, x_test = X_train[:, i], X_test[:, i]

mu, sigma = x_train.mean(), x_train.std()          # from the TRAINING set only
z_train = (x_train - mu) / sigma
z_test  = (x_test  - mu) / sigma
print(f"training mean {mu:.4f}, training std {sigma:.4f}\n")

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w, b = 0.0, 0.0
alpha = 0.5
print(" step        w         b     cost   accuracy")
for step in range(3001):
    p = sigmoid(w * z_train + b)

    if step % 500 == 0:
        cost = -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))
        acc  = ((p >= 0.5).astype(int) == y_train).mean()
        print(f"{step:5d}  {w:7.4f}  {b:8.4f}   {cost:.4f}     {acc:.4f}")

    w -= alpha * np.mean((p - y_train) * z_train)
    b -= alpha * np.mean(p - y_train)

print(f"\nfinal:  w = {w:.4f}   b = {b:.4f}")
print(f"decision boundary at z = {-b/w:.4f}, which is {mu + sigma * (-b/w):.2f} mm of mean radius")

### What the fitted model looks like

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
i = list(data.feature_names).index("mean radius")
x_train = X_train[:, i]
mu, sigma = x_train.mean(), x_train.std()

w, b = 3.5135, -0.6713                     # what the loop above found
grid = np.linspace(x_train.min(), x_train.max(), 300)
curve = 1 / (1 + np.exp(-(w * (grid - mu) / sigma + b)))

plt.figure(figsize=(7, 4))
plt.scatter(x_train[y_train == 0], y_train[y_train == 0], s=18, alpha=0.5,
            color="#2563eb", label="benign (y = 0)")
plt.scatter(x_train[y_train == 1], y_train[y_train == 1], s=18, alpha=0.5,
            color="#dc2626", label="malignant (y = 1)")
plt.plot(grid, curve, color="#111827", lw=2, label="P(malignant)")
plt.axvline(mu + sigma * (-b / w), color="#059669", ls="--", lw=1.5,
            label="boundary, 14.8 mm")
plt.axhline(0.5, color="#9ca3af", ls=":", lw=1)
plt.xlabel("mean radius"); plt.ylabel("P(malignant)")
plt.title("One feature, 455 training tumours")
plt.legend(loc="center right", fontsize=8)
plt.show()

### Scoring it on the 114 patients it never saw

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
i = list(data.feature_names).index("mean radius")
mu, sigma = X_train[:, i].mean(), X_train[:, i].std()
z_test = (X_test[:, i] - mu) / sigma

w, b = 3.5135, -0.6713
p_test = 1 / (1 + np.exp(-(w * z_test + b)))
pred = (p_test >= 0.5).astype(int)

print("confusion matrix, rows = truth, columns = prediction")
print(confusion_matrix(y_test, pred))
print("        pred benign   pred malignant")
print(" benign      TN             FP")
print(" malignant   FN             TP")
print(f"\nROC AUC: {roc_auc_score(y_test, p_test):.4f}")
print()
print(classification_report(y_test, pred, target_names=["benign", "malignant"]))

## One knob per feature

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler().fit(X_train)     # 30 means and 30 stds, from train only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

n, m = Z_train.shape
w, b = np.zeros(m), 0.0                    # one weight per feature now
alpha = 0.5
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    w -= alpha * (Z_train.T @ (p - y_train) / n)
    b -= alpha * np.mean(p - y_train)

p_test = sigmoid(Z_test @ w + b)
pred = (p_test >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print(f"\nROC AUC: {roc_auc_score(y_test, p_test):.4f}\n")
print(classification_report(y_test, pred, target_names=["benign", "malignant"]))

## Example 2: the same problem, in scikit-learn

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

model = LogisticRegression(max_iter=5000)      # 1. choose
model.fit(Z_train, y_train)                    # 2. train   <- the loop above
proba = model.predict_proba(Z_test)[:, 1]      # 3. predict, as probabilities
pred  = model.predict(Z_test)                  #    or as labels

print(confusion_matrix(y_test, pred))
print(f"\nROC AUC: {roc_auc_score(y_test, proba):.4f}\n")
print(classification_report(y_test, pred, target_names=["benign", "malignant"]))
print(f"one coefficient per feature: {len(model.coef_[0])}, plus an intercept of {model.intercept_[0]:.4f}")

### Do they agree patient by patient?

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

n, m = Z_train.shape
w, b = np.zeros(m), 0.0
for step in range(1000):
    p = sigmoid(Z_train @ w + b)
    w -= 0.5 * (Z_train.T @ (p - y_train) / n)
    b -= 0.5 * np.mean(p - y_train)
ours = (sigmoid(Z_test @ w + b) >= 0.5).astype(int)

theirs = LogisticRegression(max_iter=5000).fit(Z_train, y_train).predict(Z_test)

print("patients where the two models disagree:", (ours != theirs).sum(), "of", len(y_test))

## Reporting it for a cancer screen

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, recall_score, precision_score

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=5000).fit(scaler.transform(X_train), y_train)
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

print(f"{'threshold':>10} {'FN (missed)':>12} {'FP (alarms)':>12} {'recall':>8} {'precision':>10}")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"{t:10.1f} {fn:12d} {fp:12d} {recall_score(y_test, pred):8.4f} "
          f"{precision_score(y_test, pred):10.4f}")

Back to the [Topic 9b lecture note](https://dongchul.kim/csci3320-fall2026/topic/9b).